# Analysis I - Observed workloads
**Evidence class: observational.** This notebook measures properties of a published sensor log. It does not measure internal server packets, watts, or causal energy savings.

## Questions and methods
1. **How variable are logged arrivals?** Count records by calendar day and source hour; examine per-sensor interarrival gaps.
2. **How complete is the source?** Audit timestamps, roster IDs, missing channels and each sensor's representation before selecting replay intervals.
3. **Can these data establish communication or energy savings?** No: the fields contain readings and log times, not service edges, bytes or power.

**Original source:** [Intel Lab Data](https://db.lcs.mit.edu/labdata/labdata.html), Intel Berkeley Research Lab, 28 February-5 April 2004. The published log has about 2.3 million records from 54 sensors. The [dataset register](sources.md) records the other sources planned for service graph and power context. Download the original `data.txt.gz` to `../data/raw/intel/` relative to this notebook.


In [ ]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

SOURCE = 'https://db.lcs.mit.edu/labdata/labdata.html'
PATH = Path('../data/raw/intel/data.txt.gz')
assert PATH.exists(), f'Download the original archive from {SOURCE} to {PATH}'
sha256 = hashlib.sha256(PATH.read_bytes()).hexdigest()
print('Input SHA-256:', sha256)


## Parse and audit
The original file uses a single space delimiter. Empty channel values must remain empty; splitting on arbitrary whitespace can shift columns without an error. Keep the raw record count and exclusions visible. Source times have no asserted timezone.


In [ ]:
columns = ['date','time','epoch','moteid','temperature','humidity','light','voltage']
raw = pd.read_csv(PATH, sep=' ', names=columns, header=None, compression='gzip', low_memory=False)
raw['timestamp'] = pd.to_datetime(raw['date'].astype(str) + ' ' + raw['time'].astype(str), errors='coerce')
for col in columns[2:]: raw[col] = pd.to_numeric(raw[col], errors='coerce')
audit = pd.Series({
    'raw records':len(raw), 'invalid timestamps':int(raw.timestamp.isna().sum()),
    'missing sensor ID':int(raw.moteid.isna().sum()),
    'out-of-roster ID':int((raw.moteid.notna() & ~raw.moteid.between(1,54)).sum()),
    'at least one missing channel':int(raw[['temperature','humidity','light','voltage']].isna().any(axis=1).sum())})
display(audit.to_frame('records'))
events = raw.loc[raw.timestamp.notna() & raw.moteid.between(1,54), ['timestamp','moteid']].sort_values('timestamp',kind='stable')
display(Markdown(f'**Technical:** {len(events):,} records have valid timestamps and roster IDs. Missing values or long gaps cannot be attributed to network loss without packet logs.  \n**Plain language:** We can count what was recorded; we cannot tell why something is absent.'))


## Arrival rate and coverage
Endpoint days are partial. Even an interior day can include logging downtime; the charts describe the *recorded* workload and do not establish normal demand.


In [ ]:
daily = events.groupby(events.timestamp.dt.floor('D')).size().rename('readings')
hourly = events.groupby(events.timestamp.dt.hour).size().reindex(range(24),fill_value=0)
per_mote = events.groupby('moteid').size().reindex(range(1,55),fill_value=0)
fig = px.line(x=daily.index,y=daily.values,labels={'x':'Source calendar day','y':'Logged readings'},title='1. Recorded volume by day'); fig.show()
fig = px.bar(x=hourly.index,y=hourly.values,labels={'x':'Source hour (across all dates)','y':'Logged readings'},title='2. Hour-of-day totals, without uptime correction'); fig.show()
fig = px.bar(x=per_mote.index,y=per_mote.values,labels={'x':'Sensor ID','y':'Logged readings'},title='3. Unequal sensor coverage'); fig.show()
interior = daily.iloc[1:-1]
display(Markdown(f'**Technical:** Interior-day volume ranges from {interior.min():,} to {interior.max():,}; sensor counts range from {per_mote.min():,} to {per_mote.max():,}. Low counts may reflect collection outages, and hour totals are not normalized by uptime.  \n**Plain language:** Recorded activity varies sharply, but we should check that sensors were working before calling a quiet day a low-traffic day.'))


## Interarrival gaps
Calculate intervals separately for each sensor. Report full-distribution quantiles alongside a histogram trimmed for legibility. This avoids letting gaps between *different* sensors masquerade as a single sensor's cadence.


In [ ]:
gaps = events.groupby('moteid').timestamp.diff().dt.total_seconds().dropna()
gaps = gaps[gaps >= 0]
counts,edges = np.histogram(gaps[gaps <= gaps.quantile(.99)], bins=90)
fig = px.bar(x=(edges[:-1]+edges[1:])/2,y=counts,labels={'x':'Seconds between logged readings from one sensor','y':'Frequency'},title='4. Within-sensor gap frequency (upper 1% omitted from view)'); fig.show()
probabilities = np.linspace(0,1,101)
fig = px.line(x=gaps.quantile(probabilities).to_numpy(),y=probabilities,labels={'x':'Gap in seconds','y':'Cumulative proportion'},title='5. Gap quantiles, including the upper tail'); fig.show()
display(Markdown(f'**Technical:** Median within-sensor gap: {gaps.median():.1f} s; p95: {gaps.quantile(.95):.1f} s. These are conditional on observed records.  \n**Plain language:** A typical sensor reading was logged about half a minute after its previous one; some recorded gaps were much longer.'))


## Interpretation and next dataset
These plots answer arrival and coverage questions for one historical wireless deployment. They do not measure east-west traffic. The primary follow-up is the [Alibaba 2021 microservice trace](https://github.com/alibaba/clusterdata/tree/master/cluster-trace-microservices-v2021): reconstruct caller-to-callee paths, stratify by call type, and disclose its 0.5% call-graph sample and large download size. The [assessment](assessment.md) lists evidence gaps. Continue in [Analysis II](02_models_and_experiments.ipynb).
